developing a composite indicator which is a single
number that can be used to compare countries, people or cities.
An indicator is a quantitative or qualitative measure derived from a series of observed
facts that can reveal relative positions (e.g. of a country) in a given area. When
evaluated at regular intervals, an indicator can point out the direction of change across
different units and through time.

Topic : Women's Empowerment and Economic Development
Comparing WEI across regions.

In [60]:
import pandas as pd
import numpy as np

In [61]:
world_bank = pd.read_csv("data/raw/world_bank_indicators.csv")

world_bank.head()

world_bank.columns

Index(['Country Name', 'Country Code', 'Series Name', 'Series Code',
       '2018 [YR2018]', '2019 [YR2019]', '2020 [YR2020]', '2021 [YR2021]',
       '2022 [YR2022]', '2023 [YR2023]'],
      dtype='str')

In [62]:
# World Bank missing value markerreplacement
world_bank = world_bank.replace("..", pd.NA)


year_columns = [
    "2023 [YR2023]",
    "2022 [YR2022]",
    "2021 [YR2021]",
    "2020 [YR2020]",
    "2019 [YR2019]",
    "2018 [YR2018]"
]

# Keeping only year columns that exist in the dataset
year_columns = [col for col in year_columns if col in world_bank.columns]

# Convert year columns to numeric
for col in year_columns:
    world_bank[col] = pd.to_numeric(world_bank[col], errors="coerce")

# Using the most recent available value between 2018 and 2023
world_bank["latest_value"] = world_bank[year_columns].bfill(axis=1).iloc[:, 0]

world_bank.head()

,Country Name,Country Code,Series Name,Series Code,2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],latest_value
0,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,413.757895
1,Afghanistan,AFG,"School enrollment, secondary, female (% gross)",SE.SEC.ENRR.FE,41.350712,NaN,NaN,NaN,NaN,45.159691,45.159691
2,Afghanistan,AFG,"School enrollment, tertiary, female (% gross)",SE.TER.ENRR.FE,5.020660,NaN,5.936540,NaN,NaN,NaN,5.936540
3,Afghanistan,AFG,"Labor force participation rate, female (% of f...",SL.TLF.CACT.FE.ZS,19.805000,18.298000,16.474000,14.616000,5.159000,5.145000,5.145000
4,Afghanistan,AFG,"Unemployment, female (% of female labor force)...",SL.UEM.TOTL.FE.ZS,14.787000,15.569000,16.778000,16.946000,26.742000,26.597000,26.597000


In [63]:
world_bank_pivot = world_bank.pivot_table(
    index=["Country Name", "Country Code"],
    columns="Series Name",
    values="latest_value",
    aggfunc="first"
).reset_index()

world_bank_pivot.head()

Series Name,Country Name,Country Code,"Adolescent fertility rate (births per 1,000 women ages 15-19)",GDP per capita (current US$),"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)","Life expectancy at birth, female (years)","Maternal mortality ratio (modeled estimate, per 100,000 live births)",Proportion of seats held by women in national parliaments (%),"School enrollment, secondary, female (% gross)","School enrollment, tertiary, female (% gross)","Unemployment, female (% of female labor force) (modeled ILO estimate)"
0,Afghanistan,AFG,64.068,413.757895,5.145,67.536,521.0,27.016129,45.159691,5.936540,26.597
1,Albania,ALB,12.789,9730.869219,57.963,81.446,7.0,35.714286,92.668293,77.031527,10.957
2,Algeria,DZA,8.698,5370.477235,14.338,77.696,62.0,7.862408,103.220718,67.258843,21.064
3,American Samoa,ASM,34.181,18017.458938,NaN,75.838,NaN,NaN,NaN,NaN,NaN
4,Andorra,AND,3.480,46812.426101,NaN,86.107,11.0,50.000000,99.468742,68.003820,NaN


In [64]:
aggregate_keywords = [
    "Africa", "World", "income", "Europe", "Asia", "IDA", "IBRD",
    "Middle East", "North America", "Latin America", "Caribbean",
    "Arab World", "Euro area", "OECD", "Sub-Saharan", "Small states",
    "Fragile", "Least developed", "Heavily indebted"
]

world_bank_pivot = world_bank_pivot[
    ~world_bank_pivot["Country Name"].str.contains(
        "|".join(aggregate_keywords),
        case=False,
        na=False
    )
]

world_bank_pivot.head()

world_bank_pivot.columns.tolist()

['Country Name',
 'Country Code',
 'Adolescent fertility rate (births per 1,000 women ages 15-19)',
 'GDP per capita (current US$)',
 'Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)',
 'Life expectancy at birth, female (years)',
 'Maternal mortality ratio (modeled estimate, per 100,000 live births)',
 'Proportion of seats held by women in national parliaments (%)',
 'School enrollment, secondary, female (% gross)',
 'School enrollment, tertiary, female (% gross)',
 'Unemployment, female (% of female labor force) (modeled ILO estimate)']

In [65]:
world_bank_pivot = world_bank_pivot.rename(columns={
    "GDP per capita (current US$)": "gdp_per_capita",
    "School enrollment, secondary, female (% gross)": "female_secondary_enrollment",
    "School enrollment, tertiary, female (% gross)": "female_tertiary_enrollment",
    "Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)": "female_labor_force_participation",
    "Unemployment, female (% of female labor force) (modeled ILO estimate)": "female_unemployment",
    "Proportion of seats held by women in national parliaments (%)": "women_in_parliament",
    "Life expectancy at birth, female (years)": "female_life_expectancy",
    "Maternal mortality ratio (modeled estimate, per 100,000 live births)": "maternal_mortality",
    "Adolescent fertility rate (births per 1,000 women ages 15-19)": "adolescent_fertility"
})

world_bank_pivot.head()



Series Name,Country Name,Country Code,adolescent_fertility,gdp_per_capita,female_labor_force_participation,female_life_expectancy,maternal_mortality,women_in_parliament,female_secondary_enrollment,female_tertiary_enrollment,female_unemployment
0,Afghanistan,AFG,64.068,413.757895,5.145,67.536,521.0,27.016129,45.159691,5.936540,26.597
1,Albania,ALB,12.789,9730.869219,57.963,81.446,7.0,35.714286,92.668293,77.031527,10.957
2,Algeria,DZA,8.698,5370.477235,14.338,77.696,62.0,7.862408,103.220718,67.258843,21.064
3,American Samoa,ASM,34.181,18017.458938,NaN,75.838,NaN,NaN,NaN,NaN,NaN
4,Andorra,AND,3.480,46812.426101,NaN,86.107,11.0,50.000000,99.468742,68.003820,NaN


In [66]:
missing_report = pd.DataFrame({
    "missing_values": world_bank_pivot.isna().sum(),
    "missing_percentage": world_bank_pivot.isna().mean() * 100
}).sort_values("missing_percentage", ascending=False)

missing_report

,missing_values,missing_percentage
Series Name,,
female_tertiary_enrollment,58,27.102804
female_secondary_enrollment,40,18.691589
female_labor_force_participation,30,14.018692
female_unemployment,30,14.018692
women_in_parliament,24,11.214953
maternal_mortality,23,10.747664
gdp_per_capita,5,2.336449
adolescent_fertility,0,0.000000
Country Code,0,0.000000


In [67]:
world_bank_pivot.to_csv("data/processed/world_bank_cleaned.csv", index=False)

In [68]:
missing_report.to_csv("data/processed/world_bank_missing_report.csv")